In [16]:
from pathlib import Path
import pygimli as pg  # Stelle sicher, dass das Modul pg importiert ist und verfügbar ist
from pygimli.physics import ert
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from datetime import datetime
import os
import numpy as np
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.ticker as mticker

from Hilfsfunktionen import T_corr_nach_Inversion
from Hilfsfunktionen import plotting_function
from Hilfsfunktionen import plotting_function_FTL

# Read data
file = "two_timesteps.ohm"
base_dir = Path("filtered_data")
data = []
for unterordner in base_dir.iterdir():
    if unterordner.is_dir():
        datei_pfad = unterordner / "two_timesteps.ohm" 
        if datei_pfad.exists():
            daten_objekt = pg.load(str(datei_pfad))
            data.append([unterordner.name, daten_objekt])
            print(f"Load data: {datei_pfad}")

# Create method manager
manager = []
for ts in data:
    manager.append(ert.ERTManager(ts[1], verbose=True))

Load data: filtered_data\240710\two_timesteps.ohm
Load data: filtered_data\240808\two_timesteps.ohm


In [17]:
# Create mesh
mesh = manager[0].createMesh(quality = 34, paraMaxCellSize=0.5, paraDepth=15)
#manager[0].invert(quality = 34, paraMaxCellSize=0.5, maxIter=1 ,dPhi= 0.1, paraDepth=15,lam=20)

11/05/26 - 09:58:13 - pyGIMLi - INFO - Found 2 regions.
11/05/26 - 09:58:13 - pyGIMLi - INFO - Region with smallest marker set to background (marker=1)


In [ ]:
# FTL
manager_list_tl = []
# ScaleF-Liste
scalef_list = np.linspace(0.05, 1.9, num=50)



results_tl_diff = []
chi2_dict_FTL      = []
phi_m_dict_FTL     = []
phi_d_dict_FTL     = []
rms_dict_FTL       = []
rrms_dict_FTL      = []
phi_m_zeitlich     = []


# Inversion without temperature correction
DATA = [dat[1] for dat in data]
# Full time-lapse
for scalef in scalef_list:
    fop = pg.frameworks.MultiFrameModelling(ert.ERTModelling, scalef=scalef)
    fop.setData(DATA)
    fop.setMesh(mesh)
    print(fop.mesh()) 
    dataVec = np.concatenate([data["rhoa"] for data in DATA])
    errorVec = np.concatenate([data["err"] for data in DATA])
    startModel = fop.createStartModel(dataVec)
    inv = pg.Inversion(fop=fop, startModel=startModel, verbose=True)
    model = inv.run(dataVec, errorVec, maxIter=100, lam= 12, startModel=startModel, verbose=True)
    chi2 = []
    rrms = []
    chi2.append(round(inv.chi2(),2))
    rrms.append(round(inv.relrms(),2))
    mod = np.reshape(model, [len(DATA), -1])

    # Zeitliche Roughness berechnen
    m = np.array(inv.model)          # Länge 6208
    n = m.size // 2                    # Zellen je Zeitschritt = 3104
    m1 = m[:n]                         # Schritt 1
    m2 = m[n:]                         # Schritt 2
    
    # falls m bereits log(ρ) wäre: die nächste Zeile weglassen!
    dlog = np.log(m2) - np.log(m1)     # Differenz in log(ρ)
    
    MT = np.sum(dlog**2)               # zeitliche Rauigkeit (L2)

    
    chi2_dict_FTL.append(inv.chi2())
    phi_m_zeitlich.append(MT)
    phi_d_dict_FTL.append(inv.phiData())
    phi_m_dict_FTL.append(inv.phiModel())
    rrms_dict_FTL.append(round(inv.relrms(), 2))
    rms_dict_FTL.append(round(inv.absrms(), 2))


11/05/26 - 15:49:42 - pyGIMLi - INFO - Found 2 regions.
11/05/26 - 15:49:42 - pyGIMLi - INFO - Region with smallest marker set to background (marker=1)
11/05/26 - 15:49:42 - pyGIMLi - INFO - Creating forward mesh from region infos.
11/05/26 - 15:49:42 - pyGIMLi - INFO - Creating refined mesh (H2) to solve forward task.
11/05/26 - 15:49:43 - pyGIMLi - INFO - Mesh for forward task: Mesh: Nodes: 8540 Cells: 16788 Boundaries: 12736


3104 model cells
Mesh: Nodes: 8540 Cells: 16788 Boundaries: 12736


11/05/26 - 15:49:50 - pyGIMLi - INFO - Starting inversion.


fop: <pygimli.frameworks.timelapse.MultiFrameModelling object at 0x000001AEACFE7CE0>
Data transformation: Identity transform
Model transformation: Logarithmic transform
min/max (data): 65.6/6396
min/max (error): 3%/3.02%
min/max (start model): 1951/1951
--------------------------------------------------------------------------------
inv.iter 0 ... chi² = 17958.28
--------------------------------------------------------------------------------
inv.iter 1 ... chi² =  436.64 (dPhi = 97.54%) lam: 12.0
--------------------------------------------------------------------------------
inv.iter 2 ... chi² =  147.85 (dPhi = 66.00%) lam: 12.0
--------------------------------------------------------------------------------
inv.iter 3 ... chi² =    3.34 (dPhi = 96.43%) lam: 12.0
--------------------------------------------------------------------------------
inv.iter 4 ... chi² =    1.92 (dPhi = 30.04%) lam: 12.0
--------------------------------------------------------------------------------
inv.i

11/05/26 - 15:52:53 - pyGIMLi - INFO - Found 2 regions.
11/05/26 - 15:52:53 - pyGIMLi - INFO - Region with smallest marker set to background (marker=1)
11/05/26 - 15:52:53 - pyGIMLi - INFO - Creating forward mesh from region infos.
11/05/26 - 15:52:53 - pyGIMLi - INFO - Creating refined mesh (H2) to solve forward task.


...

chi² =    1.23 (dPhi = 1.64%) lam: 12.0
################################################################################
#                Abort criterion reached: dPhi = 1.64 (< 2.0%)                 #
################################################################################


11/05/26 - 16:10:17 - pyGIMLi - INFO - Mesh for forward task: Mesh: Nodes: 8540 Cells: 16788 Boundaries: 12736


3104 model cells
Mesh: Nodes: 8540 Cells: 16788 Boundaries: 12736


11/05/26 - 16:10:18 - pyGIMLi - INFO - Starting inversion.


fop: <pygimli.frameworks.timelapse.MultiFrameModelling object at 0x000001AEAD155D50>
Data transformation: Identity transform
Model transformation: Logarithmic transform
min/max (data): 65.6/6396
min/max (error): 3%/3.02%
min/max (start model): 1951/1951
--------------------------------------------------------------------------------
inv.iter 0 ... chi² = 17958.28
--------------------------------------------------------------------------------
inv.iter 1 ... chi² =  436.62 (dPhi = 97.54%) lam: 12.0
--------------------------------------------------------------------------------
inv.iter 2 ... chi² =  149.14 (dPhi = 65.71%) lam: 12.0
--------------------------------------------------------------------------------
inv.iter 3 ... chi² =    3.32 (dPhi = 96.46%) lam: 12.0
--------------------------------------------------------------------------------
inv.iter 4 ... chi² =    1.89 (dPhi = 29.95%) lam: 12.0
--------------------------------------------------------------------------------
inv.i

In [25]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import UnivariateSpline
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D

def plot_l_curve(phi_m_list, phi_d_list, lam_list, spline_s=0.003, fontsize=14, save_path=None):
    """
    L-Curve Plot mit Curvature und Maximum Curvature Annotation.

    Parameters:
    -----------
    phi_m_list : array-like
        Liste der Modell-Roughness-Werte (Φₘ).
    phi_d_list : array-like
        Liste der Daten-Misfit-Werte (Φ_d).
    lam_list : array-like
        Liste der Regularisierungsparameter λ.
    spline_s : float
        Glättungsfaktor für UnivariateSpline (default=0.001).
    fontsize : int
        Schriftgröße für Achsen, Titel, Legende und Annotation.
    """
    fig, ax1 = plt.subplots(1, 1, figsize=(8,6))

    # ==========================
    # L-Curve Plot
    # ==========================
    ax1.plot(phi_m_list, phi_d_list, color='black', marker='o', alpha=0.5, linestyle='')

    sc = ax1.scatter(phi_m_list, phi_d_list,
                     c=lam_list,
                     cmap='jet',
                     norm=mcolors.LogNorm(vmin=min(lam_list), vmax=max(lam_list)),
                     s=60,
                     edgecolor='black',
                     zorder=3)

    cbar = fig.colorbar(sc, ax=ax1)
    cbar.set_label("Regularization parameter sf", fontsize=fontsize)
    num_ticks = 6
    tick_values = np.logspace(np.log10(min(lam_list)), np.log10(max(lam_list)), num=num_ticks)
    cbar.locator = mticker.FixedLocator(tick_values)
    cbar.ax.set_yticklabels([f"{val:.3g}" for val in tick_values], fontsize=fontsize)
    cbar.ax.minorticks_off()

    ax1.set_xlabel("Temporal model roughness Φₘ", fontsize=fontsize)
    ax1.set_ylabel("Data misfit Φ_d (χ²)", fontsize=fontsize)
    ax1.set_title("L-curve FTL", fontsize=fontsize+2)

    # Tick-Labels größer machen
    ax1.tick_params(axis='both', labelsize=fontsize)

    # ==========================
    # Krümmung berechnen
    # ==========================
    x = (phi_m_list - np.min(phi_m_list)) / (np.max(phi_m_list) - np.min(phi_m_list))
    y = (phi_d_list - np.min(phi_d_list)) / (np.max(phi_d_list) - np.min(phi_d_list))
    t = np.arange(len(x))

    spl_x = UnivariateSpline(t, x, k=3, s=spline_s*len(t))
    spl_y = UnivariateSpline(t, y, k=3, s=spline_s*len(t))

    dx_disc, ddx_disc = spl_x.derivative(1)(t), spl_x.derivative(2)(t)
    dy_disc, ddy_disc = spl_y.derivative(1)(t), spl_y.derivative(2)(t)
    curv_disc = np.abs(dx_disc*ddy_disc - dy_disc*ddx_disc) / (dx_disc**2 + dy_disc**2)**1.5

    # ==========================
    # Zweite Achse für Curvature ohne Beschriftung
    # ==========================
    ax1_curv = ax1.twinx()
    ax1_curv.set_ylabel('')               
    ax1_curv.set_yticklabels([])          
    ax1_curv.tick_params(axis='y', length=5)  

    # Kreuze auf Curvature-Höhe, letzte 10 Punkte weggeschnitten
    phi_m_plot = phi_m_list[:-15]
    curv_plot = curv_disc[:-15]

    ax1_curv.scatter(phi_m_plot, curv_plot,
                     s=60,
                     marker='x',
                     color='black')

    # ==========================
    # Eine gemeinsame Legende (Kreise + Kreuze)
    # ==========================
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', label='L-Curve', markerfacecolor='black', markersize=8, alpha=0.5, linestyle='None'),
        Line2D([0], [0], marker='x', color='black', label='Curvature', markersize=8, linestyle='None')
    ]
    ax1.legend(handles=legend_elements, loc='upper right', fontsize=fontsize)

    # ==========================
    # Maximalpunkt markieren mit Pfeil von rechts oben
    # ==========================
    idx_max_curv = np.argmax(curv_plot)
    phi_m_max = phi_m_plot[idx_max_curv]
    curv_max = curv_plot[idx_max_curv]
    lam_max = lam_list[idx_max_curv]

    ax1_curv.annotate(f"Maximum curvature\nat sf={lam_max:.3f}",
                     xy=(phi_m_max+2, curv_max),
                     xytext=(phi_m_max + 30, curv_max*0.75),
                     arrowprops=dict(facecolor='black', arrowstyle="->"),
                     ha='center',
                     color='black',
                     fontsize=fontsize)
    #
    plt.tight_layout()
    # ==========================
    # Optional speichern
    # ==========================
    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Figure saved to {save_path}")

    plt.show()


In [ ]:
plot_l_curve(phi_m_zeitlich, phi_d_dict_FTL, scalef_list, spline_s=0.001, fontsize=14, save_path='./L_curve_FTL/L-Curve_zwei_Zeitschritte_gute_Darstellung_groesser_neu.png')

np.save('./L_curve_FTL/L-Curve_zwei_Zeitschritte_phi_m_zeitlich_3.npy', phi_m_zeitlich)
np.save('./L_curve_FTL/L-Curve_zwei_Zeitschritte_phi_d_mean_new_3.npy', phi_d_dict_FTL)
np.save('./L_curve_FTL/L-Curve_zwei_Zeitschritte_scalef_list_2.npy', scalef_list)